# Análisis de stocks: métodos con 100% de hits en los últimos 5 años

Lee todos los `{Ticket}_valuaciones_hits_por_anio.xlsx` de `Valuaciones_USA/` (generados por
`Valuaciones_todas.ipynb`) y busca, por ticker y método de valuación, si en los últimos 5 años
el precio llegó a tocar la banda del objetivo en el 100% de los periodos.

Criterios usados (decididos con el usuario):
- Se piden al menos 5 periodos dentro de la ventana de 5 años para que el 100% cuente como señal.
- "Últimos 5 años" se cuenta hacia atrás desde el último periodo disponible **de cada ticker**
  (no desde la fecha de hoy), igual que el propio filtro de 10 años en `Valuaciones_todas.ipynb`.
- Si el ÚNICO periodo que falló (`hit=0`) dentro de la ventana es justo el último periodo (el
  más reciente), no cuenta en contra del hit rate: todavía puede no haberle dado tiempo de tocar
  la banda. Si el que falló es un periodo más viejo (no el último), sí cuenta como fallo real y
  el método no entra al 100%. La columna `Periodo_fallo` siempre muestra cuál fue el periodo que
  falló (y si ya se puede confirmar como fallo o sigue pendiente), aunque no haya bajado el
  Hit_rate_%.
- Nota: estos archivos ya vienen filtrados aguas arriba (solo incluyen métodos con hit rate
  >=70% en sus últimos 10 años). Un método con 100% en 5 años pero flojo entre el año 6 y el 10
  no va a aparecer aquí porque ni siquiera tiene fila en el archivo de origen.

In [9]:
import glob
import os
import re

import numpy as np
import pandas as pd

CARPETA_VALUACIONES = "Valuaciones_USA"
PATRON_ARCHIVO = "_valuaciones_hits_por_anio.xlsx"
VENTANA_ANIOS = 5
MINIMO_PERIODOS = 5

## Cargar y apilar todos los archivos de hits por año

In [10]:
def listar_archivos_hits(carpeta=CARPETA_VALUACIONES, patron=PATRON_ARCHIVO):
    """Rutas de los archivos {Ticket}_valuaciones_hits_por_anio.xlsx, sin los temporales ~$."""
    rutas = glob.glob(os.path.join(carpeta, f"*{patron}"))
    return sorted(r for r in rutas if not os.path.basename(r).startswith('~$'))


def cargar_hits(carpeta=CARPETA_VALUACIONES, patron=PATRON_ARCHIVO):
    """Concatena todos los archivos de hits por año en un solo DataFrame, con columna Ticket."""
    piezas = []
    for ruta in listar_archivos_hits(carpeta, patron):
        ticker = os.path.basename(ruta)[:-len(patron)]
        df = pd.read_excel(ruta)
        df.insert(0, 'Ticket', ticker)
        piezas.append(df)
    if not piezas:
        return pd.DataFrame(columns=['Ticket', 'Año', 'Metodo', 'Valor', 'hit', 'tth_weeks', 'hit_date', 'side'])
    return pd.concat(piezas, ignore_index=True)


hits = cargar_hits()
hits['Fecha'] = pd.to_datetime(hits['Año'], format='%b%Y') + pd.offsets.MonthEnd(0)
print(f"{hits['Ticket'].nunique()} tickers, {len(hits)} filas (Ticket x Metodo x Año)")

36 tickers, 2169 filas (Ticket x Metodo x Año)


C:\Users\mar_c\AppData\Local\Temp\ipykernel_31360\3566745275.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(piezas, ignore_index=True)
C:\Users\mar_c\AppData\Local\Temp\ipykernel_31360\3566745275.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(piezas, ignore_index=True)


## Ventana de 5 años por ticker y cálculo del hit rate por método

In [11]:
def _filas_en_ventana(hits, anios=VENTANA_ANIOS):
    """Filas de `hits` dentro de los últimos `anios` años de cada ticker (desde su último
    periodo disponible), con `hit` no nulo y convertido a entero."""
    fecha_max_ticker = hits.groupby('Ticket')['Fecha'].transform('max')
    inicio_ventana = fecha_max_ticker - pd.DateOffset(years=anios)
    en_ventana = hits[(hits['Fecha'] >= inicio_ventana) & hits['hit'].notna()].copy()
    en_ventana['hit'] = en_ventana['hit'].astype(int)
    return en_ventana


def hit_rate_ultimos_n_anios(hits, anios=VENTANA_ANIOS, minimo=MINIMO_PERIODOS):
    """Hit rate por Ticket+Metodo en los últimos `anios` años (desde el último periodo de cada
    ticker), exigiendo al menos `minimo` periodos con hit no nulo dentro de la ventana.

    Si el ÚNICO periodo que falló (hit=0) es justo el último de la ventana, no cuenta en contra:
    puede que simplemente no le haya dado tiempo de tocar la banda todavía. Si el que falló es
    un periodo más viejo (no el último), sí cuenta como fallo real y baja el hit rate.
    """
    en_ventana = _filas_en_ventana(hits, anios)

    base = (
        en_ventana.groupby(['Ticket', 'Metodo'])
        .agg(
            N=('hit', 'size'),
            Hits=('hit', 'sum'),
            Primer_periodo=('Año', 'first'),
            Semanas_promedio_hasta_hit=('tth_weeks', 'mean'),
        )
        .reset_index()
    )

    idx_ultimo = en_ventana.groupby(['Ticket', 'Metodo'])['Fecha'].idxmax()
    ultimo = en_ventana.loc[idx_ultimo, ['Ticket', 'Metodo', 'Año', 'hit']].rename(
        columns={'Año': 'Ultimo_periodo', 'hit': 'Ultimo_hit'})

    resumen = base.merge(ultimo, on=['Ticket', 'Metodo'])
    # Si el último periodo de la ventana falló, no se cuenta en contra todavía.
    resumen['Hits_ajustado'] = resumen['Hits'] + (resumen['Ultimo_hit'] == 0).astype(int)
    resumen['Hit_rate_%'] = (resumen['Hits_ajustado'] / resumen['N'] * 100).round(1)
    resumen = resumen.drop(columns=['Ultimo_hit'])

    return resumen[resumen['N'] >= minimo].sort_values(['Ticket', 'Metodo']).reset_index(drop=True)


def metodos_con_umbral(resumen, umbral):
    """Filas de `resumen` (Ticket+Metodo) con Hit_rate_% >= umbral."""
    return resumen[resumen['Hit_rate_%'] >= umbral].sort_values(
        ['Ticket', 'Metodo']).reset_index(drop=True)


def con_periodo_de_fallo(resumen, detalle):
    """Agrega la columna Periodo_fallo: el (los) periodo(s) con hit=0 real (sin el ajuste del
    último periodo), marcando si ya se puede confirmar como fallo definitivo (pasaron 156
    semanas) o sigue pendiente. Se deja visible aunque el Hit_rate_% ya no lo cuente en contra
    cuando ese periodo es el último de la ventana."""
    fallos = detalle.loc[detalle['hit'] == 0, ['Ticket', 'Metodo', 'Año', 'Fecha']].copy()
    semanas_transcurridas = (pd.Timestamp.today() - fallos['Fecha']).dt.days / 7
    fallos['Periodo_fallo'] = np.where(
        semanas_transcurridas >= 156,
        fallos['Año'] + ' (confirmado)',
        fallos['Año'] + ' (pendiente)'
    )
    fallos = fallos.groupby(['Ticket', 'Metodo'])['Periodo_fallo'].agg(', '.join).reset_index()
    return resumen.merge(fallos, on=['Ticket', 'Metodo'], how='left').fillna({'Periodo_fallo': ''})


def con_conteo_hits_side(resumen, detalle):
    """Agrega Hits_long, Hits_short y Hits_at_target: de los hits (hit=1) dentro de la ventana,
    cuántos fueron con el precio viniendo de abajo del objetivo (long), de arriba (short), o ya
    estaban dentro de la banda desde el primer periodo de la ventana (at_target). Las tres
    columnas siempre suman Hits."""
    aciertos = detalle[detalle['hit'] == 1]
    conteo = (
        aciertos.groupby(['Ticket', 'Metodo', 'side']).size()
        .unstack(fill_value=0)
        .reindex(columns=['long', 'short', 'at_target'], fill_value=0)
        .rename(columns={'long': 'Hits_long', 'short': 'Hits_short', 'at_target': 'Hits_at_target'})
        .reset_index()
    )
    resumen = resumen.merge(conteo, on=['Ticket', 'Metodo'], how='left')
    for col in ('Hits_long', 'Hits_short', 'Hits_at_target'):
        resumen[col] = resumen[col].fillna(0).astype(int)
    return resumen


resumen_5y = hit_rate_ultimos_n_anios(hits)
detalle_5y = _filas_en_ventana(hits)   # mismas filas, pero sin agrupar: una por Ticket+Metodo+Año

metodos_100 = con_conteo_hits_side(
    con_periodo_de_fallo(metodos_con_umbral(resumen_5y, 100.0), detalle_5y),
    detalle_5y
)
print(f"Tickers con al menos un método al 100% en los últimos {VENTANA_ANIOS} años "
      f"(mínimo {MINIMO_PERIODOS} periodos; un fallo solo en el último periodo no cuenta en "
      f"contra): {metodos_100['Ticket'].nunique()}")
metodos_100

Tickers con al menos un método al 100% en los últimos 5 años (mínimo 5 periodos; un fallo solo en el último periodo no cuenta en contra): 28


,Ticket,Metodo,N,Hits,Primer_periodo,Semanas_promedio_hasta_hit,Ultimo_periodo,Hits_ajustado,Hit_rate_%,Periodo_fallo,Hits_long,Hits_short,Hits_at_target
0,ACN,Intrinsic_CAGR_FCF,6,5,Aug2021,37.800000,Nov2025,6,100.0,Nov2025 (pendiente),1,3,1
1,ACN,Intrinsic_FCF_Avg,6,5,Aug2021,17.200000,Nov2025,6,100.0,Nov2025 (pendiente),0,2,3
2,ACN,Intrinsic_Mediana,6,6,Aug2021,39.333333,Nov2025,6,100.0,,3,2,1
3,ADBE,Peter_Lynch_Value_2,6,6,Nov2019,6.000000,Aug2024,6,100.0,,0,3,3
4,ADP,GF_Value_Guru,6,6,Jun2021,6.000000,Sep2025,6,100.0,,2,1,3
5,ADSK,GF_Value_Guru,6,6,Jan2021,42.166667,Oct2025,6,100.0,,2,2,2
6,AMGN,GF_Value_Guru,6,6,Dec2020,15.666667,Sep2025,6,100.0,,5,0,1
7,AMGN,Median_PS_Value_Guru,6,6,Dec2020,16.000000,Sep2025,6,100.0,,5,0,1
8,AMZN,GF_Value_Guru,6,6,Dec2020,32.166667,Sep2025,6,100.0,,2,2,2
9,BAC,Book_Value_per_Share,6,5,Dec2020,6.800000,Sep2025,6,100.0,Sep2025 (pendiente),1,2,2


## Listado de tickets (únicos) con 100%

In [12]:
tickers_100 = sorted(metodos_100['Ticket'].unique())
print(tickers_100)

['ACN', 'ADBE', 'ADP', 'ADSK', 'AMGN', 'AMZN', 'BAC', 'BRO', 'CDNS', 'CRM', 'DECK', 'FISV', 'FTNT', 'GOOG', 'HD', 'INTU', 'LMT', 'META', 'MMC', 'MNST', 'MORN', 'MSFT', 'NFLX', 'PEP', 'ROP', 'TSM', 'V', 'VRTX']
